<a href="https://colab.research.google.com/github/vzavala/StatsBook/blob/master/Python/ch3_gibbs_covariance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Setup.** Run this cell first: it locates the data files this notebook reads. On Google
Colab it downloads them (the `Data/` and `Support/` folders) from the book's GitHub
repository; on your computer, run the notebook from the folder that contains `Data/`.

In [1]:
import os
import sys
path = ''
if 'google.colab' in sys.modules and not os.path.isdir('Data'):
    if not os.path.isdir('statsbook'):
        os.system('git clone -q --depth 1 --filter=blob:none --sparse '
                  'https://github.com/vzavala/StatsBook.git statsbook')
        os.system('git -C statsbook sparse-checkout set Python/Data Python/Support')
    path = 'statsbook/Python/'

Covariance and correlation for the Gibbs reactor (Victor Z, UW-Madison, 2024).

| Slides | Python | Meaning |
|---|---|---|
| $x_\omega\in\mathbb{R}^5$, $S$ | `x_omega` (S x 5), `S` | pressure, conversion, outflows of CO, H$_2$, CH$_3$OH |
| $\hat{m}_i=\hat{\mathbb{E}}[X_i]$ | `m_hat` | empirical means |
| $\hat{\mathrm{Cov}}[X]$ | `Cov_hat` | sample covariance (with $1/S$) |
| $\hat{\mathrm{PCorr}}[X]=D^{-1}\hat{\mathrm{Cov}}[X]D^{-1}$ | `PCorr_hat`, `D` | Pearson correlation |

## Sample covariance and correlation

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

names = ['P', 'conversion', 'F_CO', 'F_H2', 'F_CH3OH']
x_omega = np.loadtxt(path+'Data/gibbs_covariance.dat')
S, n = x_omega.shape

np.random.seed(1)   # measurement noise (results differ slightly from the Matlab version)
x_omega = x_omega + np.random.normal(0, 1, (S, n)) * np.array([10, 0.1, 10, 10, 10])

fig, axs = plt.subplots(2, 2, figsize=(10, 8))
for ax, (i, j) in zip(axs.ravel(), [(0, 1), (0, 2), (2, 3), (3, 4)]):
    ax.plot(x_omega[:, i], x_omega[:, j], 'ko', markerfacecolor='w', markersize=4)
    ax.set_xlabel(names[i])
    ax.set_ylabel(names[j])
    ax.grid(True)
plt.tight_layout()
plt.show()

/var/folders/nc/c3cjtn9d173b4m3s1nfbpw100000gn/T/ipykernel_41053/499187691.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


$\hat{\mathrm{Cov}}[X]_{ij}=\frac{1}{S}\sum_{\omega}(x_{\omega,i}-\hat{m}_i)(x_{\omega,j}-\hat{m}_j)$.
(`np.cov` divides by $S-1$ unless `bias=True`.)

In [3]:
m_hat = x_omega.mean(axis=0)
Cov_hat = (x_omega - m_hat).T @ (x_omega - m_hat) / S
assert np.allclose(Cov_hat, np.cov(x_omega, rowvar=False, bias=True))

D = np.diag(np.sqrt(np.diag(Cov_hat)))
PCorr_hat = np.linalg.inv(D) @ Cov_hat @ np.linalg.inv(D)
assert np.allclose(PCorr_hat, np.corrcoef(x_omega, rowvar=False))

pd.options.display.float_format = '{:.2f}'.format
print("Cov_hat[X] =")
print(pd.DataFrame(Cov_hat, index=names, columns=names))
print("\nPCorr_hat[X] =")
print(pd.DataFrame(PCorr_hat, index=names, columns=names))
print("\neigenvalues of Cov_hat (all >= 0):", np.round(np.linalg.eigvalsh(Cov_hat), 3))

Cov_hat[X] =
                 P  conversion    F_CO    F_H2  F_CH3OH
P           701.09        1.74 -149.44 -332.61   168.97
conversion    1.74        0.02   -0.50   -1.02     0.38
F_CO       -149.44       -0.50  142.50   95.84   -37.57
F_H2       -332.61       -1.02   95.84  278.81  -102.95
F_CH3OH     168.97        0.38  -37.57 -102.95   147.41

PCorr_hat[X] =
               P  conversion  F_CO  F_H2  F_CH3OH
P           1.00        0.52 -0.47 -0.75     0.53
conversion  0.52        1.00 -0.33 -0.48     0.25
F_CO       -0.47       -0.33  1.00  0.48    -0.26
F_H2       -0.75       -0.48  0.48  1.00    -0.51
F_CH3OH     0.53        0.25 -0.26 -0.51     1.00

eigenvalues of Cov_hat (all >= 0): [1.10000e-02 8.07870e+01 1.04620e+02 1.11341e+02 9.73072e+02]


## Conditional variance of pressure and conversion

For a Gaussian $(X_1,X_2)$ (pressure, conversion):
$\Sigma_{2|1}=\Sigma_{22}-\Sigma_{21}\Sigma_{11}^{-1}\Sigma_{12}$ and
$\Sigma_{1|2}=\Sigma_{11}-\Sigma_{12}\Sigma_{22}^{-1}\Sigma_{21}$.
We use the covariance values quoted on the slide. Note that the relative reduction
$1-\Sigma_{2|1}/\Sigma_{22}=1-\Sigma_{1|2}/\Sigma_{11}=\rho^2$ is the same in both directions.

In [4]:
Sigma = np.array([[776.94, 1.83],
                  [1.83, 0.02]])     # Cov[X] for (pressure, conversion), from the slide
Sigma_2g1 = Sigma[1, 1] - Sigma[1, 0] / Sigma[0, 0] * Sigma[0, 1]
Sigma_1g2 = Sigma[0, 0] - Sigma[0, 1] / Sigma[1, 1] * Sigma[1, 0]
rho = Sigma[0, 1] / np.sqrt(Sigma[0, 0] * Sigma[1, 1])
print(f"Sigma_2|1 = {Sigma_2g1:.4f}  (reduction {1 - Sigma_2g1/Sigma[1, 1]:.0%})")
print(f"Sigma_1|2 = {Sigma_1g2:.2f}  (reduction {1 - Sigma_1g2/Sigma[0, 0]:.0%})")
print(f"rho = {rho:.2f},  rho^2 = {rho**2:.2f}")

Sigma_2|1 = 0.0157  (reduction 22%)
Sigma_1|2 = 609.50  (reduction 22%)
rho = 0.46,  rho^2 = 0.22


The same computation with the full data set: the variance of conversion given the
other four variables, $\Sigma_{1|2}$ with block $1=$ conversion and block $2=$ the rest.

In [5]:
order = [1, 0, 2, 3, 4]               # conversion first
C = Cov_hat[np.ix_(order, order)]
Sigma_11, Sigma_12, Sigma_22 = C[0, 0], C[0, 1:], C[1:, 1:]
Sigma_1g2 = Sigma_11 - Sigma_12 @ np.linalg.solve(Sigma_22, Sigma_12)
print(f"V[conversion] = {Sigma_11:.4f},  V[conversion | other 4 variables] = {Sigma_1g2:.4f}")

V[conversion] = 0.0158,  V[conversion | other 4 variables] = 0.0111
